# FIRST LAB

### If necessary, install Cursor Extensions

1. From the View menu, select Extensions
2. Search for Python
3. Click on "Python" made by "ms-python" and select Install if not already installed
4. Search for Jupyter
5. Click on "Jupyter" made by "ms-toolsai" and select Install if not already installed


### Next Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import *
from IPython.display import Markdown, display
from huggingface_hub import login
from tqdm.notebook import tqdm
from openai import OpenAI
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import requests
import torch

c:\Users\Karan\Desktop\SLMs\slm_engineering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check for GPU / CPU
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = "cuda" if torch.cuda.is_available() else "cpu"

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA: 12.8


# Connecting to Hugging Face

The next cell is where we load in the environment variables in your `.env` file and connect to Hugging Face.  

In [3]:
# Log in to HuggingFace - if you get a "Note" about Environment variable being set, ignore it

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Let's make a quick call to a SLM model to get started!

In [4]:
# Model details:
model_id = "meta-llama/Llama-3.2-3B-Instruct"

In [5]:
# Load model directly
tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct").to(device)

In [6]:
# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load model directly onto available hardware
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Model loaded successfully!")
print("Memory footprint:", round(model.get_memory_footprint() / 1024**3, 2), "GB")

W1018 13:34:10.095000 24416 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\Karan\Desktop\SLMs\slm_engineering\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 2/2 [00:13<00:00,  6.72s/it]


Model loaded successfully!
Memory footprint: 2.05 GB


In [7]:
# Defining prompt & message
messages=[
        {"role": "system", "content": "You are a helpful, concise AI assistant."},
        {"role": "user", "content": "Explain the concept of an SLM (Small Language Model) in one paragraph."}
    ]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(device)

In [8]:
# Output
outputs = model.generate(**inputs, max_new_tokens=150)

# Extract Response
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:], 
    clean_up_tokenization_spaces=False)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
c:\Users\Karan\Desktop\SLMs\slm_engineering\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [9]:
# Extract Output
print(response)

A Small Language Model (SLM) is a type of artificial intelligence (AI) model designed to process and generate human language, but with a significantly smaller size and computational requirements compared to larger language models. SLMs are typically used in applications where smaller models can suffice, such as chatbots, text classification, or language translation. They are trained on a smaller dataset and have fewer parameters, making them more efficient and easier to deploy. Despite their smaller size, SLMs can still be effective in tasks that require basic language understanding and generation, such as answering simple questions, generating short text responses, or performing language translation.<|eot_id|>


## Onwards with Website Scrapping

In [10]:
# Define our system prompt"

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [11]:
# This function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [12]:
# Let's try out this utility

def summarize(url):

    # Optional: Add headers to mimic a real browser to avoid being blocked
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        # Raise an exception for HTTP error codes (4xx or 5xx)
        response.raise_for_status()
        
        # Print the HTML content of the page
        website = response.text
        
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

    # print(messages_for(website))
    
    inputs = tokenizer.apply_chat_template(
        conversation = messages_for(website),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    # Output
    outputs = model.generate(**inputs, max_new_tokens=200)

    return print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], clean_up_tokenization_spaces=False))


In [13]:
# Try this out, and then try for a few more websites
url = "https://karanoberoi28.github.io"
display(Markdown(summarize(url)))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Summary**
This website appears to be a personal blog/site of Karan Oberoi, a data scientist. The site has various sections, including an "About" section, a "Projects" section, a "Blogs" section, and a "SLMs" (Supply Chain Management Systems) section.

**News/Announcements**
No recent news or announcements are found in the provided HTML. However, there are some placeholder sections that indicate upcoming content, such as the "Projects" section being rebuilt and the "SLMs" section being under construction.

**Content Analysis**
The website seems to be well-structured and easy to navigate. The use of a consistent layout and design throughout the site is a plus. The content is mostly informative, providing an overview of Karan's work and interests. However, some sections, such as the "Blogs" section, are currently empty and indicate that the site is still under development.

**Technical Notes**
The website uses HTML5, CSS3


<IPython.core.display.Markdown object>

In [14]:
# Clear GPU memory and model
del outputs
del inputs
del model

torch.cuda.empty_cache()
torch.cuda.ipc_collect()